# 🔄 ReAct Prompting Pattern

**Day 3 — Notebook 3 of 4 | Estimated Time: 25 minutes**

---

## What You'll Learn
- The ReAct (Reason + Act) pattern for tool-augmented prompting
- How to simulate tool usage through prompt design
- Thought → Action → Observation loops
- Why ReAct is the foundation for AI agents (Day 10 preview)

---

## What is ReAct?

**ReAct** combines **Reasoning** (chain-of-thought) with **Acting** (taking actions like searching, calculating, etc.).

The loop:
```
Thought: I need to find X...
Action: Search["X"]
Observation: X is...
Thought: Now I know X, I need to calculate Y...
Action: Calculate[Y]
Observation: Y = ...
Thought: I now have enough information.
Answer: ...
```

This is exactly how modern AI agents work — and you can simulate it through prompting alone!

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

from google.genai import errors

try:
    client = genai.Client(api_key=get_api_key())
except errors.APIError as e:
    print(f"Gemini API Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
else:
    MODEL_ID = "gemini-2.5-flash"
    print("✅ Ready!")

---

## 1. Simulating ReAct Through Prompts

Let's teach the model to reason and act step by step.

In [ ]:
react_prompt = """
You are a helpful research assistant. Answer questions by thinking step by step.
For each step, use this format:

Thought: [what you're thinking or what you need to find out]
Action: [the action you'd take, e.g., Search, Calculate, Lookup]
Observation: [what you found/know]

Continue this loop until you can give a final answer.
End with: Answer: [your final answer]

Question: What would be the total cost of flying a team of 5 people from 
New York to London, if the average round-trip ticket costs $800 and each 
person needs a $150/night hotel for 3 nights? Don't forget to add a 15% 
contingency buffer for unexpected expenses.
"""

response = client.models.generate_content(model=MODEL_ID, contents=react_prompt)
Markdown(response.text)

---

## 2. ReAct with Simulated Tools

We can define "tools" the model can call and simulate the observations.

In [ ]:
# Define simulated tools
def simulated_search(query: str) -> str:
    """Simulate a knowledge base search."""
    knowledge_base = {
        "company vacation policy": "Employees get 20 days PTO per year. Unused days carry over up to 5 days.",
        "remote work policy": "Employees can work remotely up to 3 days per week with manager approval.",
        "expense policy": "Business expenses over $500 require VP approval. Meals capped at $50/person.",
        "parental leave": "12 weeks paid parental leave for all new parents.",
    }
    for key, value in knowledge_base.items():
        if key in query.lower():
            return value
    return "No relevant information found."

def simulated_calculate(expression: str) -> str:
    """Simulate a calculator."""
    try:
        result = eval(expression)  # Simple eval for demo only
        return str(result)
    except:
        return "Error in calculation"


# Multi-step ReAct with actual tool calls
question = "If I've used 12 vacation days this year, how many do I have left, and can I carry over any to next year?"

# Step 1: Ask the model what to do
step1_prompt = f"""You are an HR assistant. Answer questions using these tools:
- Search[query]: Search the company knowledge base
- Calculate[expression]: Perform a calculation

Question: {question}

What is your first thought and action? Use the format:
Thought: ...
Action: ...
"""

response1 = client.models.generate_content(model=MODEL_ID, contents=step1_prompt)
print("Step 1:")
print(response1.text)

# Step 2: Provide the observation
observation = simulated_search("company vacation policy")
step2_prompt = f"""Continue from where we left off.

Question: {question}

{response1.text}
Observation: {observation}

Continue your reasoning. What is your next thought and action?
If you have enough information, provide the final Answer.
"""

response2 = client.models.generate_content(model=MODEL_ID, contents=step2_prompt)
print("\nStep 2:")
print(response2.text)

---

## 3. Full ReAct Loop Implementation

Let's build a complete ReAct loop that automatically dispatches tool calls.

In [ ]:
import re

def react_agent(question: str, max_steps: int = 5) -> str:
    """A simple ReAct agent that reasons and acts."""
    
    tools_description = """Available tools:
    - Search[query]: Search the company knowledge base
    - Calculate[expression]: Perform a mathematical calculation
    
    Use this format for each step:
    Thought: [your reasoning]
    Action: ToolName[argument]
    
    When you have the final answer:
    Thought: I now have enough information.
    Answer: [your final answer]"""
    
    context = f"{tools_description}\n\nQuestion: {question}\n"
    
    for step in range(max_steps):
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=context,
            config=types.GenerateContentConfig(temperature=0),
        )
        result = response.text.strip()
        context += result + "\n"
        print(f"--- Step {step + 1} ---")
        print(result)
        
        # Check if we have a final answer
        if "Answer:" in result:
            return result.split("Answer:")[-1].strip()
        
        # Parse and execute action
        action_match = re.search(r'Action:\s*(\w+)\[(.+?)\]', result)
        if action_match:
            tool_name = action_match.group(1)
            tool_arg = action_match.group(2)
            
            if tool_name == "Search":
                observation = simulated_search(tool_arg)
            elif tool_name == "Calculate":
                observation = simulated_calculate(tool_arg)
            else:
                observation = f"Unknown tool: {tool_name}"
            
            context += f"Observation: {observation}\n"
            print(f"Observation: {observation}\n")
    
    return "Max steps reached without an answer."


# Test the agent
answer = react_agent("How many vacation days can I carry over if I've used 17 days this year?")
print(f"\n🎯 Final Answer: {answer}")

---

## 🏋️ Exercise: Extend the ReAct Agent

Add a new tool to the ReAct agent and test it.

In [ ]:
# Exercise: Add a "Lookup" tool for employee data
# TODO:
# 1. Create a simulated_lookup function with employee data
#    (e.g., {"Alice": {"department": "Engineering", "hire_date": "2022-01-15"}, ...})
# 2. Add it to the react_agent function
# 3. Test with: "When did Alice join the company, and how many vacation days 
#    would she have earned by now?"

# Your code here

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| ReAct: Synergizing Reasoning and Acting | Paper | [arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629) |
| ReAct Prompting Guide | Blog | [promptingguide.ai/techniques/react](https://www.promptingguide.ai/techniques/react) |

---

**Next up:** [04_Building_a_Prompt_Library.ipynb](./04_Building_a_Prompt_Library.ipynb)